# EDUAI Model Factory · Kaggle · Fase 1
**Objetivo:** comprobar el entrenamiento QLoRA de EDUAI Nano (350M), sin tocar producción.

**Importante:** este notebook es un **smoke test** con 15 ejemplos sintéticos; no demuestra mejoras del modelo. Mantén el notebook **privado**. No agregues credenciales, conversaciones ni datos personales. Antes de iniciar: Settings → Accelerator → GPU y activar Internet para obtener dependencias.

In [ ]:
import os, subprocess
from pathlib import Path
WORK = Path("/kaggle/working")
REPO = WORK / "eduai-platform"
BRANCH = "feat/model-lab-kaggle-phase1"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://github.com/innova-space-edu/eduai-platform.git", str(REPO)], check=True)
os.chdir(REPO)
subprocess.run(["git", "rev-parse", "--short", "HEAD"], check=True)

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-r",
                "training/eduai-local/requirements.txt"], check=True)

In [ ]:
import torch
assert torch.cuda.is_available(), "Activa una GPU en Settings > Accelerator"
print("GPU:", torch.cuda.get_device_name(0))
print("Torch:", torch.__version__)

## Validación sin gastar entrenamiento
El preflight verifica el JSONL y que estén presentes los archivos de fábrica. No descarga modelos.

In [ ]:
subprocess.run([sys.executable, "training/eduai-local/kaggle/smoke_train.py", "--dry-run"], check=True)
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s",
                "training/eduai-local/kaggle", "-p", "test_*.py", "-v"], check=True)

## Entrenamiento QLoRA mínimo
Esta celda **sí usa GPU y descarga el modelo base**. Ejecutar una sola vez por prueba. Si falla, conserva el error y el reporte, sin reintentar automáticamente.

In [ ]:
subprocess.run([sys.executable, "training/eduai-local/kaggle/smoke_train.py",
                "--epochs", "1"], check=True)

## Evidencia del resultado
Solo si terminó sin errores, revisar el reporte y guardar el adaptador. No publicar ni subirlo automáticamente a EDUAI.

In [ ]:
import json
from pathlib import Path
out = Path("/kaggle/working/eduai-nano-smoke-lora")
report = json.loads((out / "kaggle-smoke-report.json").read_text(encoding="utf-8"))
print(json.dumps(report, ensure_ascii=False, indent=2))
assert report["status"] == "adapter-created"
print("Archivos principales:", [p.name for p in out.iterdir() if p.is_file()])

In [ ]:
import zipfile
archive = Path("/kaggle/working/eduai-nano-smoke-adapter.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for f in out.iterdir():
        if f.is_file() and f.suffix in (".json", ".safetensors", ".bin", ".model"):
            z.write(f, f.name)
print("ZIP privado:", archive, "bytes:", archive.stat().st_size)

## Próxima fase
Este adaptador es solo evidencia técnica de que el entrenamiento se ejecutó. Falta crear un dataset representativo con separación train/validation/test, entrenar nuevamente, fusionar pesos, convertir a GGUF, comparar contra el modelo base y superar Quality Gate. Mantener todo fuera de producción.